<a href="https://colab.research.google.com/github/tsubasa-iino/psi4book/blob/main/compchem_book_ch06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 6章 分子の構造を最適化してみよう


### 環境構築

#### Google Colab上にPsi4をインストール

In [1]:
!pip install -q condacolab
import condacolab
import os

# バグ回避パッチ
if "LD_LIBRARY_PATH" not in os.environ:
    os.environ["LD_LIBRARY_PATH"] = ""

print("Installing CondaColab (Base)...")
condacolab.install()

Installing CondaColab (Base)...
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:14
🔁 Restarting kernel...


ランタイム｜セッションを再起動する｜はい を実行。

In [ ]:
import condacolab
import os
import sys

condacolab.check()

# 1. 邪魔なPinningを削除
if os.path.exists("/usr/local/conda-meta/pinned"):
    !rm /usr/local/conda-meta/pinned

# 2. Python 3.12 と Psi4 をインストール（ディスク書き換え）
print("Upgrading Python to 3.12 & Installing Psi4...")
!mamba install -y -q python=3.12 psi4 -c conda-forge/label/libint_dev -c conda-forge

# 3. Pinningの復元（成功環境の再現）
os.makedirs("/usr/local/conda-meta", exist_ok=True)
with open("/usr/local/conda-meta/pinned", "w") as f:
    f.write("python 3.12.*\n")

# 4. 【最重要】カーネルの自殺（強制再起動）
# これにより、メモリ上のPython 3.11を殺し、ディスク上のPython 3.12をロードさせます
print("\n🔄 RESTARTING KERNEL TO LOAD PYTHON 3.12...")
import time
time.sleep(1)
os.kill(os.getpid(), 9)

✨🍰✨ Everything looks OK!
Upgrading Python to 3.12 & Installing Psi4...
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done

🔄 RESTARTING KERNEL TO LOAD PYTHON 3.12...


ランタイム｜セッションを再起動する｜はい を実行。

In [1]:
import sys
import os

# パスが通っていなければ通す
target_path = "/usr/local/lib/python3.12/site-packages"
if target_path not in sys.path:
    sys.path.insert(0, target_path)

import psi4
print(f"✅ Restart Successful.")
print(f"Psi4 Version: {psi4.__version__}")
print(f"Python Version: {sys.version.split()[0]}") # ここが3.12になっているはず

# 計算テスト
psi4.set_memory('500 MB')
mol = psi4.geometry("O\nH 1 0.96\nH 1 0.96 2 104.5")
en = psi4.energy('scf/cc-pvdz')
print(f"Energy: {en:.6f}")

✅ Restart Successful.
Psi4 Version: 1.10
Python Version: 3.12.12
Energy: -76.026633


ここまででインストール確認完了。

In [2]:
import os
import datetime
import numpy as np
import pandas as pd
import psi4

print(f'current time: {datetime.datetime.now()}')
print(f'python version:\n{sys.version}')
print(f'numpy version: {np.__version__}')
print(f'pandas version: {pd.__version__}')
print(f'psi4 version: {psi4.__version__}')

current time: 2026-01-26 06:13:48.289667
python version:
3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy version: 2.0.2
pandas version: 2.2.2
psi4 version: 1.10


#### 必要なライブラリと関数の定義

In [3]:
# 三次元構造可視化のためのライブラリ
!pip install py3Dmol
import py3Dmol


def show_3D(mol: psi4.core.Molecule) -> py3Dmol.view:
    """
    Psi4のMoleculeオブジェクトをpy3Dmolで描画する
    Args:
        mol: 描画対象の分子

    Returns:
        py3Dmol.view: py3Dmolの描画オブジェクト

    """
    view = py3Dmol.view(width=400, height=400)
    xyz = mol.save_string_xyz_file()
    view.addModel(xyz, 'xyz')
    view.setStyle({'stick': {}})
    view.setBackgroundColor('#e1e1e1')
    view.zoomTo()

    return view.show()

#### 計算資源の設定

In [4]:
# 計算資源の確認（CPU, RAM）
!cat /proc/cpuinfo

processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2199.998
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed bhi its
bogomips	: 4399.99
clflush size	: 64
cache_alignment	: 64
address sizes

In [5]:
!cat /proc/meminfo

MemTotal:       13286964 kB
MemFree:         8533452 kB
MemAvailable:   11815788 kB
Buffers:          215292 kB
Cached:          3041112 kB
SwapCached:            0 kB
Active:          1514028 kB
Inactive:        2703508 kB
Active(anon):       4668 kB
Inactive(anon):   967880 kB
Active(file):    1509360 kB
Inactive(file):  1735628 kB
Unevictable:           8 kB
Mlocked:               8 kB
SwapTotal:             0 kB
SwapFree:              0 kB
Dirty:              1268 kB
Writeback:             0 kB
AnonPages:        961316 kB
Mapped:           523924 kB
Shmem:             11408 kB
KReclaimable:     362992 kB
Slab:             422596 kB
SReclaimable:     362992 kB
SUnreclaim:        59604 kB
KernelStack:        5464 kB
PageTables:        18532 kB
SecPageTables:         0 kB
NFS_Unstable:          0 kB
Bounce:                0 kB
WritebackTmp:          0 kB
CommitLimit:     6643480 kB
Committed_AS:    3074372 kB
VmallocTotal:   34359738367 kB
VmallocUsed:       11944 kB
VmallocChunk:    

In [6]:
n_cpu = os.cpu_count()
ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024 ** 3)

In [7]:
# 環境に応じて計算資源を設定
psi4.set_num_threads(n_cpu)
psi4.set_memory(f'{ram * 0.9: .0f}GB')

11000000000

### 6.1 分子の構造を最適化するとはどういうことだろうか

#### 水分子の計算

In [8]:
psi4.set_output_file('h2o_opt_hf-sto3g.log')

PosixPath('h2o_opt_hf-sto3g.log')

##### ① 座標の並進と回転がある場合

In [9]:
# 水分子の構造の設定
h2o = psi4.geometry('''
0 1
O       -0.7520847362      0.3573098626      0.0156794168
H        0.2372416200      0.3907947487     -0.0110698848
H       -1.0414501312      1.0972341870     -0.5754069255
''')

print(h2o.save_string_xyz())  # この時点で勝手に座標並進回転が行われる

h2o_fin = h2o.clone()   # 最適化する水分子
h2o_ini = h2o.clone()   # initial構造のままの水分子

# 構造最適化計算の実行
psi4.optimize('hf/sto-3g', molecule=h2o_fin)

0 1
 O    0.000000000000   -0.000000000000   -0.067840817916
 H    0.783036648261    0.000000000000    0.538341505515
 H   -0.783036648261   -0.000000000000    0.538341505515

Optimizer: Optimization complete!


-74.96599011058093

In [10]:
from psi4.driver.qcdb import Molecule

# B787で整列
rmsd, mill, aligned_ini = Molecule.B787(
    concern_mol=h2o_ini,  # initial構造を最適化したfinal構造になるべく整列させる。h2o_iniオブジェクトには影響を与えない。
    ref_mol=h2o_fin,
    atoms_map=True,
    mols_align=False
)  # Final RMSD =   0.0247 [A]と出力される

# Å単位に変換
bohr2angstroms = psi4.constants.bohr2angstroms

print()
# 方法1: .np accessorを使用
aligned_ini1 = aligned_ini.geometry().np * bohr2angstroms
print("整列後のinitial座標1:")
print(aligned_ini1)

print()
# 方法2: to_array()メソッドを使用
aligned_ini2 = aligned_ini.geometry().to_array() * bohr2angstroms
print("整列後のinitial座標2:")
print(aligned_ini2)

print()
# 方法3: np.array()で変換
aligned_ini3 = np.array(aligned_ini.geometry()) * bohr2angstroms
print("整列後のinitial座標3:")
print(aligned_ini3)

print()
# 原子ごとの変位を計算
fin_coords = h2o_fin.geometry().to_array() * bohr2angstroms
displacements = fin_coords - aligned_ini1
print("原子ごとの変位 [Ang]:")
for i, disp in enumerate(displacements):
    print(f"原子 {i+1} ({aligned_ini.symbol(i)}): {disp}")

print()
print("変位のノルム")
print(np.linalg.norm(displacements))

Start RMSD =   0.0297 [A] (naive)
<<<  trial        1  [ 0  1  2] yields RMSD 0.02473407  >>>
Total time [s] for      1 iterations: 0.000916
Hungarian time [s] for atom ordering: 0.000235
Kabsch time [s] for mol alignment:    0.000681
Final RMSD =   0.0247 [A]
Mirror match: False
AlignmentMill(shift=array([ 0.00000000,  0.00000000, -0.03102795]), rotation=array([[ 1.00000000,  0.00000000,  0.00000000],
       [ 0.00000000,  1.00000000,  0.00000000],
       [ 0.00000000,  0.00000000,  1.00000000]]), atommap=array([ 0,  1,  2]), mirror=False)

整列後のinitial座標1:
[[ 0.00000000  0.00000000 -0.05142153]
 [ 0.78303665  0.00000000  0.55476079]
 [-0.78303665 -0.00000000  0.55476079]]

整列後のinitial座標2:
[[ 0.00000000  0.00000000 -0.05142153]
 [ 0.78303665  0.00000000  0.55476079]
 [-0.78303665 -0.00000000  0.55476079]]

整列後のinitial座標3:
[[ 0.00000000  0.00000000 -0.05142153]
 [ 0.78303665  0.00000000  0.55476079]
 [-0.78303665 -0.00000000  0.55476079]]

原子ごとの変位 [Ang]:
原子 1 (O): [-0.00000000  0.000000

In [11]:
# 最適化構造の出力
print(h2o_fin.save_string_xyz())

0 1
 O   -0.000000000000    0.000000000000   -0.071153222115
 H    0.758023512289    0.000000000000    0.564626634700
 H   -0.758023512289   -0.000000000000    0.564626634700



In [12]:
# 最適化前後の構造の描画
show_3D(h2o_ini)
show_3D(h2o_fin)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

##### ② 座標の並進と回転がない場合

In [13]:
# 水分子の構造の設定
h2o_fixed = psi4.geometry('''
0 1
O       -0.7520847362      0.3573098626      0.0156794168
H        0.2372416200      0.3907947487     -0.0110698848
H       -1.0414501312      1.0972341870     -0.5754069255
no_com
no_reorient
''')

print(h2o_fixed.save_string_xyz())  # no_com, no_reorientのおかげで座標並進回転されない

h2o_fixed_fin = h2o_fixed.clone()   # 最適化する水分子
h2o_fixed_ini = h2o_fixed.clone()   # initial構造のままの水分子

# 構造最適化計算の実行
psi4.optimize('hf/sto-3g', molecule=h2o_fixed_fin)

0 1
 O   -0.752084736200    0.357309862600    0.015679416800
 H    0.237241620000    0.390794748700   -0.011069884800
 H   -1.041450131200    1.097234187000   -0.575406925500

Optimizer: Optimization complete!


-74.96599011057806

In [14]:
from psi4.driver.qcdb import Molecule

# B787で整列
rmsd, mill, aligned_fixed_ini = Molecule.B787(
    concern_mol=h2o_fixed_ini,  # initial構造を最適化したfinal構造になるべく整列させる
    ref_mol=h2o_fixed_fin,
    atoms_map=True,
    mols_align=False
)  # Final RMSD =   0.0247 [A]と出力される（先程と同じ結果になる）

# Å単位に変換
bohr2angstroms = psi4.constants.bohr2angstroms

print()
# 方法1: .np accessorを使用
aligned_fixed_ini1 = aligned_fixed_ini.geometry().np * bohr2angstroms
print("整列後のinitial座標1:")
print(aligned_fixed_ini1)

print()
# 方法2: to_array()メソッドを使用
aligned_fixed_ini2 = aligned_fixed_ini.geometry().to_array() * bohr2angstroms
print("整列後のinitial座標2:")
print(aligned_fixed_ini2)

print()
# 方法3: np.array()で変換
aligned_fixed_ini3 = np.array(aligned_fixed_ini.geometry()) * bohr2angstroms
print("整列後のinitial座標3:")
print(aligned_fixed_ini3)

print()
# 原子ごとの変位を計算
fixed_fin_coords = h2o_fixed_fin.geometry().to_array() * bohr2angstroms
fixed_displacements = fixed_fin_coords - aligned_fixed_ini1
print("原子ごとの変位 [Ang]:")
for i, disp in enumerate(fixed_displacements):
    print(f"原子 {i+1} ({aligned_fixed_ini.symbol(i)}): {disp}")


print()
print("変位のノルム")
print(np.linalg.norm(fixed_displacements))  # 先程と同じ結果になる

Start RMSD =   0.0247 [A] (naive)
<<<  trial        1  [ 0  1  2] yields RMSD 0.02473407  >>>
Total time [s] for      1 iterations: 0.00177
Hungarian time [s] for atom ordering: 0.000357
Kabsch time [s] for mol alignment:    0.00141
Final RMSD =   0.0247 [A]
Mirror match: False
AlignmentMill(shift=array([ 0.00000000,  0.00000000, -0.00000000]), rotation=array([[ 1.00000000, -0.00000000,  0.00000000],
       [ 0.00000000,  1.00000000,  0.00000000],
       [-0.00000000, -0.00000000,  1.00000000]]), atommap=array([ 0,  1,  2]), mirror=False)

整列後のinitial座標1:
[[-0.75208474  0.35730986  0.01567942]
 [ 0.23724162  0.39079475 -0.01106988]
 [-1.04145013  1.09723419 -0.57540693]]

整列後のinitial座標2:
[[-0.75208474  0.35730986  0.01567942]
 [ 0.23724162  0.39079475 -0.01106988]
 [-1.04145013  1.09723419 -0.57540693]]

整列後のinitial座標3:
[[-0.75208474  0.35730986  0.01567942]
 [ 0.23724162  0.39079475 -0.01106988]
 [-1.04145013  1.09723419 -0.57540693]]

原子ごとの変位 [Ang]:
原子 1 (O): [-0.01139213 -0.01258753

In [15]:
# 最適化構造の出力
print(h2o_fixed_fin.save_string_xyz())

0 1
 O   -0.763476863601    0.344722337353    0.025734923847
 H    0.222514571617    0.408371677738   -0.025111162200
 H   -1.015330955417    1.092244783209   -0.571421155147



In [16]:
# 最適化前後の構造の描画
show_3D(h2o_fixed_ini)
show_3D(h2o_fixed_fin)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### 異なる終了判定基準で計算

In [ ]:
psi4.set_output_file('h2o_opt_hf-sto3g_2.log')
psi4.set_options({'g_convergence': 'GAU_VERYTIGHT'})
psi4.optimize('hf/sto-3g', molecule=h2o_fin)

Optimizer: Optimization complete!


-74.96599012254234

In [26]:
# 最適化構造の出力
print(h2o_fixed_fin.save_string_xyz())

0 1
 O   -0.763476863018    0.344722337993    0.025734923335
 H    0.222514571183    0.408371677496   -0.025111162006
 H   -1.015330955565    1.092244782811   -0.571421154829



In [ ]:
print(h2o_fin.save_string_xyz())

0 1
 O    0.000000000000    0.000000000000   -0.071157472511
 H    0.758086102577    0.000000000000    0.564660363141
 H   -0.758086102577   -0.000000000000    0.564660363141



In [ ]:
psi4.core.clean_options()

In [ ]:
from pathlib import Path
from IPython.display import display, HTML
import html  # 文字のエスケープ用

files = ["h2o_opt_hf-sto3g.log", "h2o_opt_hf-sto3g_2.log"]

cols = []
for f in files:
    text = Path(f).read_text()
    cols.append(f"""
        <td style="vertical-align: top;">
          <div style="
              max-height: 500px;
              max-width: 800px;
              overflow: auto;
              white-space: pre;
              font-family: monospace;
              font-size: 11px;">
            <b>{f}</b>
            <br>
            {html.escape(text)}
          </div>
        </td>
    """)

html_str = "<table><tr>" + "\n".join(cols) + "</tr></table>"
display(HTML(html_str))

### 6.2 局所解と全体最適解の違いを知ろう

#### 1,2-ジクロロエタン

##### ゴーシュ配座

In [ ]:
# ログファイルを指定
psi4.set_output_file('DCE_gauche.log')

PosixPath('DCE_gauche.log')

In [ ]:
# gauche配座の分子を定義
dce_gauche = psi4.geometry('''
0 1
C       -4.7587744478     -0.1541943645     -0.0559642548
C       -3.2346420969     -0.1669054169      0.0013710608
H       -5.1443980180     -1.1552590965      0.2305585920
H       -5.0986095260      0.0746426192     -1.0881859644
Cl      -5.4379603530      1.0500010949      1.0719697870
H       -2.8490185267     -0.9530856920     -0.6813781716
H       -2.8948070187     -0.3957485023      1.0335914176
Cl      -2.5554561917      1.4011538588     -0.5119631531
''')

In [ ]:
# 構造最適化
psi4.optimize('hf/sto-3g', molecule=dce_gauche)

Optimizer: Optimization complete!


-986.3008339045542

In [ ]:
print(dce_gauche.save_string_xyz())

0 1
 C    0.037762166399    0.770163219439   -0.971537824733
 C   -0.037762166399   -0.770163219439   -0.971537824733
 H   -0.410203105309    1.151657992326   -1.890882946527
 H    1.071722420300    1.110299167686   -0.905504473920
CL   -0.872199685220    1.495297274966    0.413989079670
 H    0.410203105309   -1.151657992326   -1.890882946527
 H   -1.071722420300   -1.110299167686   -0.905504473920
CL    0.872199685220   -1.495297274966    0.413989079670



In [ ]:
show_3D(dce_gauche)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 最適化構造をXYZファイルとして保存
dce_gauche.save_xyz_file('dce_gauche.xyz', True)

##### アンチ配座

In [ ]:
# ログファイルを指定
psi4.set_output_file('DCE_anti.log')

PosixPath('DCE_anti.log')

In [ ]:
# anti配座の分子を定義
dce_anti = psi4.geometry('''
0 1
C       -4.7565813125     -0.0715651139     -0.0312842044
C       -3.2368352322     -0.0032837663      0.0312840387
H       -5.1034202802     -1.0830926821      0.2687419304
H       -5.1034205619      0.1391627311     -1.0651107286
Cl      -5.4702549705      1.1317615051      1.0713658164
Cl      -2.5231615744     -1.2066103447     -1.0713660266
H       -2.8899959804     -0.2140116390      1.0651105567
H       -2.8899962667      1.0082438103     -0.2687420691
''')

In [ ]:
# 構造最適化
psi4.optimize('hf/sto-3g', molecule=dce_anti)

Optimizer: Optimization complete!


-986.3030468694051

In [ ]:
print(dce_anti.save_string_xyz())

0 1
 C   -0.058726039525    0.768934041844    0.000000003022
 C    0.058726039525   -0.768934041844    0.000000003022
 H   -0.579756384339    1.121607202127    0.890593554508
 H   -0.579756390644    1.121607202134   -0.890593544963
CL    1.598479157021    1.504167758266   -0.000000001312
CL   -1.598479157021   -1.504167758266   -0.000000001312
 H    0.579756384339   -1.121607202127    0.890593554508
 H    0.579756390644   -1.121607202134   -0.890593544963



In [ ]:
show_3D(dce_anti)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 最適化構造をXYZファイルとして保存
dce_anti.save_xyz_file('dce_anti.xyz', True)

#### *N*-メチルアセトアミド

##### トランス型

In [ ]:
# ログファイルを指定
psi4.set_output_file('amide_trans.log')

PosixPath('amide_trans.log')

In [ ]:
# trans型アミドを定義
amide_trans = psi4.geometry('''
0 1
C       -3.7098485198     -0.3873709111      0.0075198021
C       -2.2161209604     -0.5156039410      0.0174313797
H       -4.1174176889     -0.8479074919     -0.9167016561
H       -4.0140291615      0.6808053138      0.0390995026
H       -4.1344285447     -0.9062342937      0.8923386237
O       -1.5258331167      0.4935451068      0.0564569839
N       -1.6406778483     -1.7443753645     -0.0167727078
H       -2.2310225548     -2.5885815756     -0.0495401112
C       -0.2091301745     -1.9100381894     -0.0086522692
H        0.0239171398     -1.3703365067     -0.7563056337
H       -0.1502044449     -1.2104184268      0.9847325102
H       -0.1084673908     -2.4996883295     -0.0623493244
''')

In [ ]:
# 構造最適化計算
psi4.optimize('hf/sto-3g', molecule=amide_trans)

Optimizer: Optimization complete!


-243.86163854585817

In [ ]:
print(amide_trans.save_string_xyz())

0 1
 C   -1.893761607653    0.364723901679    0.066800935727
 C   -0.353753144327    0.359100780309   -0.019845405387
 H   -2.315226735744   -0.155580912257   -0.788396824326
 H   -2.254529236554    1.387679174865    0.077987671177
 H   -2.224130678638   -0.136646697375    0.972453119603
 O    0.336911571509    1.362241542052   -0.028467750011
 N    0.203536886429   -0.962404279856   -0.168409292460
 H   -0.343200668428   -1.665808520689    0.342095512544
 C    1.650387480404   -1.086484092036    0.098507315337
 H    2.186111114764   -0.371178379416   -0.522743698364
 H    1.910756696281   -0.896067183943    1.142968799921
 H    1.975067710454   -2.092053677577   -0.164624214999



In [ ]:
show_3D(amide_trans)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 最適構造をxyzファイルとして保存
amide_trans.save_xyz_file('amide_trans.xyz', True)

In [ ]:
# psi4.freq('hf/sto-3g', molecule=amide_trans)  # この章では説明していないので不要

##### シス型

In [ ]:
# ログファイルを指定
psi4.set_output_file('amide_cis.log')

PosixPath('amide_cis.log')

In [ ]:
# 計算オプションを指定
psi4.set_options({'FULL_HESS_EVERY': 0})

In [ ]:
# cis型アミドを定義
amide_cis = psi4.geometry('''
0 1
C       -1.8393520698      0.5395360054      0.0201416776
C       -0.3360665111      0.5163043932     -0.0431352068
H       -2.2621883973     -0.0970101712     -0.7851531402
H       -2.2211245617      1.5739335314     -0.1177698104
H       -2.1824800965      0.1717236862      1.0098177487
O        0.2533796400      1.5709822841     -0.2272103410
N        0.3883166009     -0.6304019690      0.0963770346
C       -0.1804129128     -1.9455644721      0.3185298065
H        1.4169544822     -0.5706164842      0.0417502268
H       -1.2868042702     -1.9408789470      0.3669547691
H        0.2053650601     -2.3536998162      1.2759269597
H        0.1261793516     -2.6206949891     -0.5074292733
''')

In [ ]:
# 構造最適化計算
psi4.optimize('hf/sto-3g', molecule=amide_cis)

Optimizer: Optimization complete!


-243.85946291922332

In [ ]:
print(amide_cis.save_string_xyz())

0 1
 C   -1.516298202587    0.527239409409   -0.093918198650
 C    0.024560108474    0.558782661072   -0.111057587964
 H   -1.893596211118   -0.218509363181   -0.787572835696
 H   -1.892620177401    1.501056017199   -0.388990142428
 H   -1.877225614783    0.289850443435    0.902103563985
 O    0.675596125066    1.496113513600   -0.539837951879
 N    0.682445318312   -0.572134058083    0.504292640692
 C    0.130819632943   -1.924606275932    0.276511149767
 H    1.684600446746   -0.540793689677    0.275743744623
 H   -0.865928329922   -1.998326645260    0.706577359114
 H    0.771149017783   -2.645615259000    0.781981452709
 H    0.073508774753   -2.197759191875   -0.780800996584



In [ ]:
show_3D(amide_cis)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 最適構造をxyzファイルとして保存
amide_cis.save_xyz_file('amide_cis.xyz', True)

In [ ]:
# psi4.freq('hf/sto-3g', molecule=amide_cis)  # この章では説明していないので不要

### 6.3 構造最適化のコツについて知ろう

#### 徐々に最適化レベルを上げていく

In [ ]:
psi4.core.clean_options()

In [ ]:
def record_optimization(mol: psi4.core.Molecule, theory: str) -> float:
    """
    分子を指定計算レベルで最適化する時間を計測する
    Args:
        mol: 目的とする分子
        theory: 計算レベル

    Returns:
        float: 最適化にかかった時間

    """
    import time
    start = time.time()
    psi4.optimize(theory, molecule=mol)
    end = time.time()

    return end - start

In [ ]:
# 2つの方法用に同じ初期構造で分子を2つ準備
hcho = psi4.geometry('''
0 1
 C                 -0.70124482    0.39419087    0.00000000
 O                  0.52607218    0.39419087    0.00000000
 H                 -1.29338982    1.33359487    0.00000000
 H                 -1.29338982   -0.54521313    0.00003900
 ''')

hcho2 = hcho.clone()

In [ ]:
psi4.set_output_file('hcho-low-to-high.log')

PosixPath('hcho-low-to-high.log')

In [ ]:
# 計算レベルの設定
level1 = 'mp2/cc-pvdz'
level2 = 'mp2/aug-cc-pvqz'

In [ ]:
# 低レベルでまず最適化してから高精度で最適化
time_1_1 = record_optimization(hcho, level1)
time_1_2 = record_optimization(hcho, level2)
print(hcho.save_string_xyz())
print(f'level 1: {time_1_1: .2f} sec\tlevel 2: {time_1_2: .2f} sec')
print(f'level 1 + 2: {time_1_1 + time_1_2: .2f} sec')
print('####')

Optimizer: Optimization complete!
Optimizer: Optimization complete!
0 1
 C   -0.000103729801    0.604439882275    0.000000000000
 O    0.000096407028   -0.602269590382    0.000000000000
 H   -0.000147478262    1.180746663716    0.933922519883
 H   -0.000147478262    1.180746663716   -0.933922519883

level 1:  7.18 sec	level 2:  111.27 sec
level 1 + 2:  118.45 sec
####


In [ ]:
# 直接高精度で最適化
time_2 = record_optimization(hcho2, level2)
print(hcho2.save_string_xyz())
print(f'level 2: {time_2: .2f} sec')
print('####')

Optimizer: Optimization complete!
0 1
 C   -0.000013375568    0.604462795456    0.000000000000
 O    0.000012987475   -0.602321380902    0.000000000000
 H   -0.000023430026    1.181021228621    0.933558701565
 H   -0.000023430026    1.181021228621   -0.933558701565

level 2:  159.81 sec
####


In [ ]:
# 2つの方法で得た構造の差分を計算
for i in range(hcho.natom()):
    vec1 = hcho.xyz(i)
    vec2 = hcho2.xyz(i)
    dist = vec1.distance(vec2)
    symbol = hcho.symbol(i)
    print(f'{symbol}\t{dist: .3f}')

C	 0.000
O	 0.000
H	 0.001
H	 0.001


#### 一部を固定化して最適化する

##### ジクロロエタン

In [ ]:
dce_constrained = psi4.geometry('''
0 1
 C                 -2.25311207    1.17427384    0.00000000
 H                 -1.69945761    1.74831286   -0.71333438
 H                 -3.29070391    1.18574460   -0.26109690
 C                 -1.73979635   -0.27765831    0.00000000
 H                 -2.29345264   -0.85169798   -0.71333244
 H                 -1.86707706   -0.70095780    0.97443171
 Cl                -2.04374927    1.87054265    1.60280285
 Cl                -0.03310419   -0.29652555   -0.42947162
''')

In [ ]:
show_3D(dce_constrained)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
psi4.core.clean_options()
psi4.p4util.prepare_options_for_set_options()

{}

In [ ]:
psi4.set_output_file('DCE_frozen_dihedral.log')
psi4.set_options({'FROZEN_DIHEDRAL': '7 1 4 8'})
psi4.optimize('hf/sto-3g', molecule=dce_constrained)

Optimizer: Optimization complete!


-986.2992669522881

In [ ]:
show_3D(dce_constrained)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
print(dce_constrained.save_string_xyz())

0 1
 C   -0.938521345849    0.488703277727   -0.414042869913
 H   -0.409494793572    1.099399748444   -1.145758578767
 H   -2.004804785145    0.495077703967   -0.645184802637
 C   -0.420548196967   -0.969188319623   -0.418207994626
 H   -0.961712485398   -1.543859348178   -1.171415832617
 H   -0.553767298641   -1.439951171077    0.555887579198
CL   -0.754894674097    1.269845200908    1.209533947473
CL    1.334535210521   -1.064919256932   -0.854580559660



In [ ]:
psi4.p4util.prepare_options_for_set_options()

{'FROZEN_DIHEDRAL': '7 1 4 8', 'SCF__INTS_TOLERANCE': 1e-12}

In [ ]:
psi4.core.clean_options()

In [ ]:
psi4.optimize('hf/sto-3g', molecule=dce_constrained)

Optimizer: Optimization complete!


-986.3008338953247

In [ ]:
show_3D(dce_constrained)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
print(dce_constrained.save_string_xyz())

0 1
 C   -1.060742252109    0.442530123651   -0.466326954718
 H   -0.673452640401    1.024872097499   -1.303038193731
 H   -2.151094741314    0.423357632643   -0.513306752815
 C   -0.525745623622   -1.003416586081   -0.505187668574
 H   -0.936607624901   -1.514983005622   -1.377503787733
 H   -0.805307052108   -1.547672361312    0.397492853522
CL   -0.601089983431    1.296506631672    1.061031810035
CL    1.277121579185   -1.057502618741   -0.647051601224



##### ウレア

In [ ]:
psi4.core.clean_options()

In [ ]:
urea = psi4.geometry('''
0 1
C  -1.0482368367  -0.0012423218  -0.7758260561
O  -0.0156753108  -0.0009116043  -1.4283127954
N  -1.7018904431   1.1779245825  -0.3361582586
N  -1.7266400660  -1.1964224417  -0.3989998481
H  -2.6423326976   1.1099656587   0.0477575562
H  -2.6243470466  -1.1132273634   0.0768581166
C  -1.0331079014   2.4327891484  -0.2240686746
C  -1.7709147156   3.6047187295  -0.1240375937
C   0.3646086960   2.4973476556  -0.1752142056
C  -1.1050846099   4.8208129312  -0.0180761207
H  -2.8520156513   3.5729985722  -0.1396804543
C   1.0061712280   3.7072816831  -0.0381144239
H   0.9419000890   1.5862284330  -0.2376082475
C   0.2701868058   4.8845111549   0.0085960416
H   0.7715659747   5.8417509737   0.0568813942
C  -1.0357160633  -2.4408328658  -0.2531541833
C  -1.7721762481  -3.6146113788  -0.1538842773
C   0.3585515106  -2.4973265719  -0.1766237776
C  -1.1060839791  -4.8222638807  -0.0202426056
H  -2.8530368758  -3.5848424540  -0.1873777316
C   1.0045107741  -3.7088549649  -0.0250016816
H   0.9322000371  -1.5820970571  -0.2346267401
C   0.2710808092  -4.8842783600   0.0289372341
H   0.7704073129  -5.8416948859   0.0936212507
C   2.5536477648   3.7773576262  -0.0016853817
C  -1.9225668536   6.1368680035  -0.0023114781
C   2.5550174644  -3.7772234847  -0.0001177715
C  -1.9236673895  -6.1367948329   0.0021592828
F  -1.0990604049   7.2059841426  -0.0034260349
F  -2.7134666506   6.1869873422   1.0987995779
F  -2.7140404800   6.1853874684  -1.1034015072
F   2.9670204746   5.0647872239  -0.0000217451
F   3.0409557981   3.1550722476  -1.1003739505
F   3.0396455561   3.1576990486   1.1017473456
F   3.0403564876  -3.1561078437  -1.0998936671
F   3.0397742797  -3.1567134866   1.1024612903
F   2.9667580606  -5.0647949842   0.0019500673
F  -1.0988989956  -7.2058336061   0.0039753596
F  -2.7133083352  -6.1853211090   1.1043007201
F  -2.7137953394  -6.1873520690  -1.0982323098
''')

In [ ]:
show_3D(urea)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# optkingだと正規表現にバグがあったのでgeometricを用いて拘束条件付き構造最適化を実行
!conda install -c conda-forge geometric

Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - done

# All requested packages already installed.



In [ ]:
# geomeTRIC用のオプション設定
geometric_keywords = {
    'coordsys': 'tric',  # TRIC座標系（推奨）
    'constraints': {
        'freeze': [
            {'type': 'xyz', 'indices': list(range(24, 40))}  # optking: 1-based indexing, geomeTRIC: 0-based indexing
        ]
    }
}

print(geometric_keywords)

{'coordsys': 'tric', 'constraints': {'freeze': [{'type': 'xyz', 'indices': [24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]}]}}


In [ ]:
psi4.set_output_file('urea_constrained.log')
psi4.set_options({'geom_maxiter': 100})
psi4.optimize('hf/sto-3g', molecule=urea, engine='geometric', optimizer_keywords=geometric_keywords)

-1998.3913444024647

In [ ]:
print('### Constrained optimization')
psi4.p4util.prepare_options_for_set_options()  # 上記でoptkingではなくgeomeTRICを使用したので拘束条件の設定は表示されない

### Constrained optimization


{'GEOM_MAXITER': 100, 'SCF__INTS_TOLERANCE': 1e-12}

In [ ]:
print(urea.save_string_xyz())

0 1
 C   -1.074465988403    0.026581554553   -0.351676618128
 O   -0.136497903230    0.095138714855   -1.122347208939
 N   -1.645690384431    1.153471289713    0.355091497493
 N   -1.752576669900   -1.188797603239   -0.021056220419
 H   -2.667393533234    1.204249891665    0.288248406072
 H   -2.424490233104   -1.086174599391    0.745137298024
 C   -0.993900528039    2.426287127722    0.226234813069
 C   -1.734023686354    3.601628677416    0.186612301805
 C    0.400553739145    2.494138588567    0.219142661084
 C   -1.077897278687    4.821520669332    0.124749771074
 H   -2.815575402406    3.564174532272    0.212207852283
 C    1.027715932196    3.718897602170    0.131267094604
 H    0.989487887851    1.590387620085    0.275728547867
 C    0.297977018044    4.892189021132    0.083687654866
 H    0.797838017280    5.849828418915    0.028950325646
 C   -1.011120098435   -2.427480966292    0.026817425740
 C   -1.740537331236   -3.612037148051    0.061487262119
 C    0.381779715878   -2.4

In [ ]:
show_3D(urea)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
psi4.core.clean_options()

In [ ]:
print('### Clear constraints')
psi4.p4util.prepare_options_for_set_options()

### Clear constraints


{}

In [ ]:
psi4.set_options({'geom_maxiter': 200})  # OptimizationConvergenceError: Could not converge geometry optimization in 50 iterations. となったので最大最適化サイクル数を増加。
psi4.optimize('hf/sto-3g', molecule=urea, engine='geometric')

-1998.4081187197248

In [ ]:
print('### Clear constraints')
psi4.p4util.prepare_options_for_set_options()

### Clear constraints


{'GEOM_MAXITER': 200, 'SCF__INTS_TOLERANCE': 1e-12}

In [ ]:
print(urea.save_string_xyz())

0 1
 C   -0.501414907858    0.006383340879    0.104720347069
 O    0.713551783734    0.006301919145    0.081646597931
 N   -1.331868007625    1.166319709926    0.310167017742
 N   -1.339216032563   -1.153450017661   -0.069048532826
 H   -2.240576166333    1.091567600754   -0.154593025703
 H   -2.229606159354   -1.078603362672    0.429895412244
 C   -0.793846366914    2.499531411817    0.280340077577
 C   -1.584868012482    3.533074971973   -0.220168857004
 C    0.467013085925    2.794883953250    0.800696509396
 C   -1.120214964719    4.838113102717   -0.205525150666
 H   -2.569581543015    3.316402049763   -0.614307859823
 C    0.921969092092    4.103869651970    0.800898428243
 H    1.087445690513    2.004574661077    1.199773598079
 C    0.137111877596    5.132386522896    0.299342206048
 H    0.497116888036    6.153535092401    0.313626528326
 C   -0.800606841803   -2.486726537365   -0.059701858938
 C   -1.572165340180   -3.520197104082    0.470471704565
 C    0.439543364558   -2.7

In [ ]:
show_3D(urea)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### 正確なヘッセ行列を求める

In [ ]:
# 異なる値用に同じ初期構造で分子を3つ準備
psi4.set_output_file('full_hess_every.log')
hcho = psi4.geometry('''
0 1
 C                 -0.70124482    0.39419087    0.00000000
 O                  0.52607218    0.39419087    0.00000000
 H                 -1.29338982    1.33359487    0.00000000
 H                 -1.29338982   -0.54521313    0.00003900
 ''')

hcho1 = hcho.clone()
hcho2 = hcho.clone()

In [ ]:
for (i, mol) in zip([-1, 0, 1], [hcho, hcho1, hcho2]):
    psi4.core.clean_options()
    psi4.set_options({'FULL_HESS_EVERY': i})
    time = record_optimization(mol=mol, theory='hf/3-21g')
    print(f'FULL_HESS_EVERY = {i}: {time: .2f} sec')

Optimizer: Optimization complete!
FULL_HESS_EVERY = -1:  6.99 sec
Optimizer: Optimization complete!
FULL_HESS_EVERY = 0:  3.32 sec
Optimizer: Optimization complete!
FULL_HESS_EVERY = 1:  6.57 sec
